## MISA (2024-2025)
- Alohan'ny mamerina dia avereno atao Run ny notebook iray manontolo. Ny fanaovana azy dia redémarrena mihitsy ny kernel aloha (jereo menubar, safidio **Kernel$\rightarrow$Restart Kernel and Run All Cells**).

- Izay misy hoe `YOUR CODE HERE` na `YOUR ANSWER HERE` ihany no fenoina. Afaka manampy cells vaovao raha ilaina. Aza adino ny mameno references eo ambany raha ilaina.

## References
Eto ilay references rehetra no apetraka
 * Nampiasa chatgpt tamin'ny documentations sy fanaovana zavatra mbola tsy hay fa naverina nosoratana avokoa ny code

---

# Multinomial regression

In [17]:
from random import randrange
import numpy as np
from sklearn.metrics import mean_squared_error, log_loss
from sklearn.datasets import load_diabetes, load_iris, load_digits
from scipy.special import huber
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

def grad_check_sparse(f, x, analytic_grad, num_checks=12, h=1e-5, error=1e-9):
    """
    sample a few random elements and only return numerical
    in this dimensions
    """

    for i in range(num_checks):
        ix = tuple([randrange(m) for m in x.shape])

        oldval = x[ix]
        x[ix] = oldval + h  # increment by h
        fxph = f(x)  # evaluate f(x + h)
        x[ix] = oldval - h  # increment by h
        fxmh = f(x)  # evaluate f(x - h)
        x[ix] = oldval  # reset

        grad_numerical = (fxph - fxmh) / (2 * h)
        grad_analytic = analytic_grad[ix]
        rel_error = abs(grad_numerical - grad_analytic) / (
            abs(grad_numerical) + abs(grad_analytic)
        )
        print(
            "numerical: %f analytic: %f, relative error: %e"
            % (grad_numerical, grad_analytic, rel_error)
        )
        assert rel_error < error

def rel_error(x, y):
    """ returns relative error """
    return np.max(np.abs(x - y) / (np.maximum(1e-8, np.abs(x) + np.abs(y))))

In [18]:
data = load_iris()
X_train2, y_train2 = data.data, data.target

W = np.random.randn(X_train2.shape[1], 3) * 0.0001

## Naive

In [19]:
def softmax(z):
    z_stable = z - np.max(z)
    exp_z = np.exp(z_stable)
    return exp_z / np.sum(exp_z)

In [20]:
def softmax_loss_naive(W, X, y, alpha):
    """
    Softmax loss function WITH FOR LOOPS

    Inputs:
    - W: array of shape (D, C) containing weights
    - X: array of shape (N, D) containing a minibatch of data
    - y: array of shape (N,) containing training labels
    - alpha: (float) regularization 

    Returns a tuple of:
    - loss as single float
    - gradient with respect to weights W;  same shape as W
    """
    
    # Initialization
    loss = 0.0
    dW = np.zeros_like(W)
    
    # Tandremo ny numeric instability
    # YOUR CODE HERE
    m = X.shape[0]
    n = X.shape[1]
    K = W.shape[1]

    loss = 0.0
    dW = np.zeros((n, K))

    for i in range(m):
        z = np.zeros(K)
        for k in range(K):
            s = 0.0
            for j in range(n):
                s += X[i,j] * W[j,k]
            z[k] = s

        mu = softmax(z)

        # Update loss
        loss -= np.log(mu[y[i]])

        # Gradient dW
        for k in range(K):
            for j in range(n):
                indicator = 1 if y[i] == k else 0
                dW[j,k] += X[i,j] * (mu[k] - indicator)

    loss /= m
    for j in range(n):
        for k in range(K):
            dW[j,k] /= m

    # Regularization
    reg = 0.0
    for j in range(n):
        for k in range(K):
            reg += W[j,k]**2
            dW[j,k] += 2 * (alpha) * W[j,k]

    loss += alpha * reg

    return loss, dW

### Without regularization

In [21]:
loss, dW = softmax_loss_naive(W, X_train2, y_train2, 0.0)

f = lambda W: softmax_loss_naive(W, X_train2, y_train2, 0.0)[0]
grad_numerical = grad_check_sparse(f, W, dW, error=1e-7)

numerical: -0.276215 analytic: -0.276215, relative error: 6.441079e-11
numerical: 0.026160 analytic: 0.026160, relative error: 1.034944e-09
numerical: -0.166509 analytic: -0.166509, relative error: 1.648318e-10
numerical: 0.318172 analytic: 0.318172, relative error: 8.232383e-11
numerical: 0.766555 analytic: 0.766555, relative error: 7.879652e-11
numerical: 0.096438 analytic: 0.096438, relative error: 3.036196e-11
numerical: 0.026160 analytic: 0.026160, relative error: 1.034944e-09
numerical: 0.318172 analytic: 0.318172, relative error: 8.232383e-11
numerical: -0.122598 analytic: -0.122598, relative error: 3.463912e-10
numerical: -0.276215 analytic: -0.276215, relative error: 6.441079e-11
numerical: -0.029626 analytic: -0.029626, relative error: 3.678021e-09
numerical: -0.041957 analytic: -0.041957, relative error: 2.002494e-10


### With regularization

In [22]:
loss, dW = softmax_loss_naive(W, X_train2, y_train2, 2)

f = lambda W: softmax_loss_naive(W, X_train2, y_train2, 2)[0]
grad_numerical = grad_check_sparse(f, W, dW, error=1e-7)

numerical: -0.276421 analytic: -0.276421, relative error: 8.086358e-11
numerical: 0.097413 analytic: 0.097413, relative error: 4.153967e-11
numerical: -0.251906 analytic: -0.251906, relative error: 5.215485e-10
numerical: 0.097413 analytic: 0.097413, relative error: 4.153967e-11
numerical: 0.281899 analytic: 0.281899, relative error: 5.783580e-10
numerical: -0.251906 analytic: -0.251906, relative error: 5.215485e-10
numerical: 0.097413 analytic: 0.097413, relative error: 4.153967e-11
numerical: -0.122436 analytic: -0.122436, relative error: 3.314881e-10
numerical: 0.766501 analytic: 0.766501, relative error: 7.652500e-11
numerical: 0.318263 analytic: 0.318263, relative error: 7.239900e-11
numerical: -0.165907 analytic: -0.165907, relative error: 1.918623e-10
numerical: -0.251906 analytic: -0.251906, relative error: 5.215485e-10


## Vectorized

In [25]:
def softmax_loss_vectorized(W, X, y, alpha, fit_intercept=False):
    """
    Softmax loss function WITHOUT FOR LOOPS

    Inputs:
    - W: array of shape (D, C) containing weights
    - X: array of shape (N, D) containing a minibatch of data
    - y: array of shape (N,) containing training labels
    - alpha: (float) regularization 

    Returns a tuple of:
    - loss as single float
    - gradient with respect to weights W;  same shape as W
    """
    # Initialize the loss and gradient to zero.
    loss = 0.0
    dW = np.zeros_like(W)

    # YOUR CODE HERE
    m = X.shape[0]
    
    Z = X @ W
    Z_stable = Z - np.max(Z, axis=1, keepdims=True)
    exp_Z = np.exp(Z_stable)
    mu = exp_Z / np.sum(exp_Z, axis=1, keepdims=True)

    loss = -np.mean(np.log(mu[:, y]))
    loss += alpha * np.sum(W**2)

    # Gradient
    Y = np.zeros_like(mu)
    Y[np.arange(m), y] = 1
    dW = X.T @ (mu - Y) / m
    dW += 2 * alpha * W

    return loss, dW

### Without regularization

In [26]:
loss, dW = softmax_loss_vectorized(W, X_train2, y_train2, 0.0)

f = lambda W: softmax_loss_vectorized(W, X_train2, y_train2, 0.0)[0]
grad_numerical = grad_check_sparse(f, W, dW, error=1e-7)

numerical: 0.000265 analytic: -0.041957, relative error: 1.000000e+00


AssertionError: 

### With regularization

In [9]:
loss, dW = softmax_loss_vectorized(W, X_train2, y_train2, 2)

f = lambda W: softmax_loss_vectorized(W, X_train2, y_train2, 2)[0]
grad_numerical = grad_check_sparse(f, W, dW, error=1e-7)

numerical: 0.027813 analytic: 0.027813, relative error: 5.295088e-10
numerical: -0.032607 analytic: -0.032607, relative error: 4.109570e-09
numerical: -0.123035 analytic: -0.123035, relative error: 1.677036e-10
numerical: 0.094866 analytic: 0.094866, relative error: 2.388405e-10
numerical: 0.027813 analytic: 0.027813, relative error: 5.295088e-10
numerical: -0.248854 analytic: -0.248854, relative error: 5.282768e-10
numerical: -0.167874 analytic: -0.167874, relative error: 3.844875e-10
numerical: 0.280659 analytic: 0.280659, relative error: 5.088499e-10
numerical: -0.041441 analytic: -0.041441, relative error: 4.044181e-11
numerical: 0.280659 analytic: 0.280659, relative error: 5.088499e-10
numerical: -0.032607 analytic: -0.032607, relative error: 4.109570e-09
numerical: 0.318545 analytic: 0.318545, relative error: 5.959854e-12


## Gradient descent

In [14]:
class LinearModel():
    def __init__(self, fit_intercept=True):
        self.W = None
        self.fit_intercept = fit_intercept

    def train(self, X, y, learning_rate=1e-3, alpha=0, num_iters=100, batch_size=200, verbose=False):
        if self.fit_intercept:
            # YOUR CODE HERE
            X = np.insert(X, 0, 1, axis=1)
            
        N, d = X.shape
        
        C = (np.max(y) + 1) 
        if self.W is None: # Initialization
            self.W = 0.001 * np.random.randn(d, C)

        # Run stochastic gradient descent to optimize W
        
        loss_history = []
        for it in range(num_iters):
            X_batch = None
            y_batch = None
                                                               
            # Sample batch_size elements in X_batch and y_batch
            # X_batch shape is  (batch_size, d) and y_batch shape is (batch_size,)                                                                                          
            # Hint: Use np.random.choice to generate indices
            # YOUR CODE HERE
            idx = np.random.choice(X.shape[0], batch_size, replace=False)
            X_batch = X[idx]
            y_batch = y[idx]
            
            # evaluate loss and gradient
            loss, dW = self.loss(X_batch, y_batch, alpha)
            loss_history.append(loss)

            # perform parameter update                                                                
            # Update the weights w using the gradient and the learning rate.          
            # YOUR CODE HERE
            self.W -= learning_rate * dW
            
            if verbose and it % 10000 == 0:
                print("iteration %d / %d: loss %f" % (it, num_iters, loss))
                
        return loss_history

    def predict(self, X):
        pass

    def loss(self, X_batch, y_batch, reg):
        pass

class MultinomialLogisticRegressor(LinearModel):
    """ Softmax regression """

    def loss(self, X_batch, y_batch, alpha):
        return softmax_loss_vectorized(self.W, X_batch, y_batch, alpha)
    
    def predict(self, X):
        """ 
        Inputs:
        - X: array of shape (N, D) 

        Returns:
        - y_pred: 1-dimensional array of length N, each element is an integer giving the predicted class 
        """
        # YOUR CODE HERE
        if self.fit_intercept:
            X = np.insert(X, 0, values=1, axis=1)
        scores = X @ self.W
        y_pred = np.argmax(scores, axis=1)
        return y_pred

In [15]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train2 = scaler.fit_transform(X_train2)

sk_model = LogisticRegression(fit_intercept=False)
sk_model.fit(X_train2, y_train2)
sk_pred = sk_model.predict(X_train2)
sk_accuracy = accuracy_score(y_train2, sk_pred)

model = MultinomialLogisticRegressor(fit_intercept=False)
model.train(X_train2, y_train2, num_iters=75000, batch_size=64, learning_rate=1e-3, verbose=True)
pred = model.predict(X_train2)
model_accuracy = accuracy_score(y_train2, pred)

print("Accuracy scikit-learn:", sk_accuracy)
print("Accuracy gradient descent model :", model_accuracy)
assert sk_accuracy - model_accuracy < 0.01

iteration 0 / 75000: loss 1.099959
iteration 10000 / 75000: loss 0.314943
iteration 20000 / 75000: loss 0.327183
iteration 30000 / 75000: loss 0.363350
iteration 40000 / 75000: loss 0.336390
iteration 50000 / 75000: loss 0.347384
iteration 60000 / 75000: loss 0.397474
iteration 70000 / 75000: loss 0.384885
Accuracy scikit-learn: 0.86
Accuracy gradient descent model : 0.8666666666666667


In [16]:
sk_model = LogisticRegression(fit_intercept=True)
sk_model.fit(X_train2, y_train2)
sk_pred = sk_model.predict(X_train2)
sk_accuracy = accuracy_score(y_train2, sk_pred)

model = MultinomialLogisticRegressor(fit_intercept=True)
model.train(X_train2, y_train2, num_iters=75000, batch_size=64, learning_rate=1e-3, verbose=True)
pred = model.predict(X_train2)
model_accuracy = accuracy_score(y_train2, pred)

print("Accuracy scikit-learn:", sk_accuracy)
print("Accuracy gradient descent model :", model_accuracy)
assert sk_accuracy - model_accuracy < 0.02

iteration 0 / 75000: loss 1.098103
iteration 10000 / 75000: loss 0.331782
iteration 20000 / 75000: loss 0.257844
iteration 30000 / 75000: loss 0.200652
iteration 40000 / 75000: loss 0.177150
iteration 50000 / 75000: loss 0.181456
iteration 60000 / 75000: loss 0.173528
iteration 70000 / 75000: loss 0.157007
Accuracy scikit-learn: 0.9733333333333334
Accuracy gradient descent model : 0.96
